# Seeing the World in a Can of Coke: GPU runs

Everything in the project that needs a GPU: SRM training, the SRM sweep, the neural renderer, novel-view evaluation, and diffusion inpainting.
The CPU-only steps (synthetic scene generation, splatting sweeps, charts) are included only where a GPU step depends on them.

**Before running**
1. Upload the project folder to Google Drive (the `.py` files; plus `cans_data/` and, if you already have it, `cache/` for the Cans sections).
2. Runtime > Change runtime type > GPU.
3. Set `DRIVE_PROJECT` below.

Code and data are copied to the Colab disk for speed. `runs/` and `results/` are symlinked to Drive, so checkpoints and outputs survive a disconnect.
This notebook has not been executed on Colab; the commands are the ones tested on CPU with small scenes.

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/cs180_final_project"  # the project folder on your Drive
WORK = "/content/cs180_final_project"                         # local working copy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil

os.makedirs(WORK, exist_ok=True)
for path in glob.glob(f"{DRIVE_PROJECT}/*.py"):
    shutil.copy(path, WORK)

def persist(local, remote):
    """Make `local` a symlink to a folder on Drive."""
    os.makedirs(remote, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        os.symlink(remote, local)

for name in ["runs", "results"]:
    persist(f"{WORK}/{name}", f"{DRIVE_PROJECT}/{name}")

def sh(command, scene=""):
    """Run a shell command in the project folder, optionally for one scene."""
    prefix = f"SCENE={scene} " if scene else ""
    get_ipython().system(f"cd {WORK} && {prefix}{command}")

%cd $WORK
!pip -q install lpips embreex diffusers transformers accelerate
import torch
print("CUDA:", torch.cuda.is_available())

## A. Synthetic scenes (ground truth available)

Scene generation is CPU work and takes a few minutes per scene. The scene caches stay on the Colab disk; only their `runs/` and `results/` go to Drive.

In [ ]:
SCENES = {
    "one":  "--layout one --neck 0.014",
    "two":  "--layout two --neck 0.014",
    "label": "--layout two --neck 0.014 --label",
}
for name, options in SCENES.items():
    for folder in ["runs", "results"]:
        persist(f"{WORK}/scenes/{name}/{folder}", f"{DRIVE_PROJECT}/scenes_out/{name}/{folder}")
    if not os.path.exists(f"{WORK}/scenes/{name}/cache/meta.json"):
        sh(f"python synth.py --name {name} {options}")

### A1. SRM training on one setting

Mirror cans have a single material, so `--materials 1`. `--lambda-g 0` drops the GAN term, which removes most run-to-run variance.
`--no-refine` is on because in the CPU tests the refinement network absorbed about half of the reflection on mirror cans; remove it to reproduce that.
Watch `gain` in the metrics: about 1.11 (1 / 0.9 reflectance) means the panorama has converged; larger means it is still too dark, so raise `--max-steps` or `--lr-srm`.

In [ ]:
sh("python train.py srm --run srm_a --cans a --materials 1 --no-refine --lambda-g 0 --max-steps 8000 --lr-srm 5e-3", scene="two")
sh("python eval_env.py --run srm_a", scene="two")

In [ ]:
from IPython.display import Image, display
display(Image(f"{WORK}/scenes/two/results/srm_a/pano.png"))  # estimate above reference; dark blue = unobserved

### A2. SRM sweep: can A, can B, both, across view counts

One training run per setting, so this is the expensive cell: 3 settings x len(views) runs of `STEPS` steps each.
`0` means all training views. The splatting sweep (CPU, seconds) is run first for comparison.

In [ ]:
VIEWS = "8 32 0"
STEPS = 8000
TRAIN_ARGS = "--materials 1 --no-refine --lambda-g 0 --lr-srm 5e-3"

sh(f"python experiments.py sweep --views {VIEWS}", scene="two")
sh(f"python experiments.py sweep --method srm --views {VIEWS} --steps {STEPS} --train-args='{TRAIN_ARGS}'", scene="two")
for chart in ["coverage", "psnr", "psnr_shared"]:
    display(Image(f"{WORK}/scenes/two/results/sweep_srm/{chart}.png"))

### A3. One-can scene against two-can scene

In [ ]:
sh(f"python experiments.py sweep --views {VIEWS}", scene="one")
sh(f"python experiments.py sweep --method srm --views {VIEWS} --steps {STEPS} --train-args='{TRAIN_ARGS}'", scene="one")
sh("python experiments.py merge --method srm --scenes one two")
display(Image(f"{WORK}/results/merge/coverage.png"))
display(Image(f"{WORK}/results/merge/psnr.png"))

### A4. Label separation with SRM

The labelled cans have two materials (paint and bare metal), the case the splatting baseline handles badly. This trains with and without the refinement network.

In [ ]:
common = "--cans both --materials 2 --lambda-g 0 --max-steps 8000 --lr-srm 5e-3"
sh(f"python train.py srm --run label_refine {common}", scene="label")
sh(f"python train.py srm --run label_norefine {common} --no-refine", scene="label")
sh("python eval_env.py --run label_refine", scene="label")
sh("python eval_env.py --run label_norefine", scene="label")
sh("python experiments.py ablation --views 0", scene="label")

## B. Cans dataset (real data, no ground-truth environment)

Needs `cans_data/` on Drive (see `INPUTS.md`). If `cache/` is already on Drive it is copied; otherwise `prepare.py` builds it here (CPU ray tracing, about 22 min on the original machine, 12.5 GB) and copies it back.

In [ ]:
have_cache = os.path.exists(f"{DRIVE_PROJECT}/cache/meta.json")
if have_cache:
    !rsync -a --info=progress2 "$DRIVE_PROJECT/cache/" "$WORK/cache/"
    os.makedirs(f"{WORK}/cans_data", exist_ok=True)
    shutil.copy(f"{DRIVE_PROJECT}/cans_data/xy_poses.txt", f"{WORK}/cans_data")
    shutil.copy(f"{DRIVE_PROJECT}/cans_data/xy_mesh.ply", f"{WORK}/cans_data")
else:
    !rsync -a --info=progress2 "$DRIVE_PROJECT/cans_data/" "$WORK/cans_data/"
    sh("python prepare.py")
    !rsync -a --info=progress2 "$WORK/cache/" "$DRIVE_PROJECT/cache/"
sh("python fit_cans.py")   # expect radius near 0.033 and height near 0.122 for each can
!cp "$WORK/cache/cans.json" "$DRIVE_PROJECT/cache/"

### B1. The paper's SRM stage (all surfaces, all views, 40 epochs)

This is the long run. If Colab disconnects, rerun the cell with `--resume` added: epoch-based runs continue from the last checkpoint.
Runs that use `--max-steps` do not resume correctly (the step count restarts), so keep those short enough for one session.

In [ ]:
sh("python train.py srm --run srm")           # add --resume after a disconnect
sh("python eval_env.py --run srm")
display(Image(f"{WORK}/results/srm/pano.png"))

### B2. One can against two on the real data

In [ ]:
CANS_STEPS = 20000
for cans in ["a", "b", "both"]:
    sh(f"python train.py srm --run srm_{cans} --cans {cans} --lambda-g 0 --max-steps {CANS_STEPS}")
    sh(f"python eval_env.py --run srm_{cans}")
    display(Image(f"{WORK}/results/srm_{cans}/pano.png"))

In [ ]:
sh("python experiments.py sweep --views 8 32 128 0")                                   # splatting, fast
sh(f"python experiments.py sweep --method srm --views 8 32 128 --steps 8000")          # SRM, slow

### B3. Neural renderer and novel-view evaluation (paper reproduction)

Uses the `srm` run from B1. 75 epochs by default; `--resume` works here too.

In [ ]:
sh("python train.py renderer --srm-run srm")   # add --resume after a disconnect
sh("python evaluate.py --srm-run srm --renderer-run renderer")

## C. Diffusion inpainting

Fills the unobserved texels of a recovered panorama with Stable Diffusion 2 inpainting. `--input` is a folder under the scene's `results/`.
If the model download fails, the checkpoint may need a Hugging Face login (`huggingface-cli login`) or a different `--model`.

In [ ]:
sh("python splat.py --cans both --out both", scene="two")
sh("python inpaint.py --input both --method telea", scene="two")        # classical baseline
sh("python inpaint.py --input both --method diffusion", scene="two")
display(Image(f"{WORK}/scenes/two/results/both_diffusion/compare.png"))  # recovered, filled, reference

In [ ]:
sh("python walkthrough.py --input both_diffusion --room 5 4 2.7 --eye 0.8", scene="two")
from IPython.display import Video
Video(f"{WORK}/scenes/two/results/walkthrough/walkthrough.mp4", embed=True)

## Outputs

Everything is already on Drive:
- Cans: `cs180_final_project/runs/`, `cs180_final_project/results/`
- Synthetic scenes: `cs180_final_project/scenes_out/<scene>/runs|results/`